---
# Taller 1 — Dynamic Programming sobre Milan Taxi
---

Este notebook arma el MDP de Milan Taxi (`MilanTaxiEnv` + `models.build_model`), valida
que el modelo construido corresponda al ambiente, y lo resuelve con Policy Iteration y
Value Iteration (`agents.dynamic_programming`). La implementación vive en
`src/rl_project/`; este notebook la ejercita, valida e interpreta los resultados, no la
reimplementa.



Cargamos `autoreload` (para que los cambios en `src/rl_project` se reflejen sin
reiniciar el kernel) y las funciones de `rl_project` que se usan en el resto del
notebook.


In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np

from rl_project.envs import make_env
from rl_project.evaluation import rollout, evaluate_policy

## Ambiente

`MilanTaxiEnv` (`src/rl_project/envs/milan_taxi.py`) es el MDP de este taller: un taxi en
una grilla que debe recoger un pasajero en un punto y dejarlo en otro. Es configurable
por instancia (grilla, landmarks, recompensas, estocasticidad -- ver "Parámetros
configurables" al final de esta sección). Lo que sigue describe la configuración **por
defecto**, que es la que usa este notebook (`make_env()` sin argumentos).

### Espacio de estados $\mathcal{S}$

Un estado es la tupla `(row, col, pass_idx, dest_idx)`:

- `row, col` $\in \{0, \dots, 4\}$: posición del taxi en la grilla $5 \times 5$ por
  defecto (`num_rows`/`num_cols` son parámetros del ambiente). `row = 0` es la fila
  superior; la acción `SOUTH` aumenta `row`, `NORTH` lo disminuye.
- `pass_idx` $\in \{0, 1, 2, 3, 4\}$: dónde está el pasajero. `0`-`3` = esperando en el
  landmark `env.locs[pass_idx]` (por defecto: `0` Politecnico, `1` Duomo, `2` Castello,
  `3` San Siro). `4` (`env.pass_in_taxi`) = el pasajero va dentro del taxi.
- `dest_idx` $\in \{0, 1, 2, 3\}$: landmark al que quiere llegar el pasajero actual.

Con la grilla por defecto eso da $5 \times 5 \times 5 \times 4 = 500$ estados, más **un
estado absorbente** que se agrega aparte (ver la sección de transición más abajo). De
esos 500, 100 combinan `pass_idx == dest_idx` y nunca ocurren en la práctica --
`_spawn_new_passenger` siempre sortea un origen distinto del destino -- pero se dejan en
el modelo por simplicidad; no cambian el resultado de PI/VI.

### Espacio de acciones $\mathcal{A}$

$$\mathcal{A} = \{\,\text{SOUTH}=0,\ \text{NORTH}=1,\ \text{EAST}=2,\ \text{WEST}=3,\ \text{PICKUP}=4,\ \text{DROPOFF}=5\,\}$$

Las 6 acciones están disponibles en todo estado y no cambian con la configuración.

### Recompensa $r(s, a)$

Depende solo de $(s,a)$, incluso cuando la transición es estocástica: para las acciones
de movimiento, resbalar o no resbalar cuesta lo mismo. Valores por defecto (`step_reward`,
`invalid_reward`, `dropoff_reward` del ambiente):

- $-1$ en toda acción de movimiento, incluso si el taxi choca contra el borde de la
  grilla o una pared interna (no se mueve, pero igual cuesta $-1$).
- $-1$ si `PICKUP` es válido (mismo costo que un paso normal).
- $-10$ si `PICKUP` o `DROPOFF` son inválidos.
- $+20$ si `DROPOFF` es válido (pasajero en el taxi y taxi en `env.locs[dest_idx]`).

### Dinámica de transición $p(s' \mid s, a)$

Un `DROPOFF` exitoso termina el episodio (`terminated=True`); el próximo pasajero y
destino se sortean recién en el siguiente `reset()`. La tarea es **episódica**: cada
episodio empieza con un pasajero y termina cuando se lo deja en su destino.

La función `env._movement_outcomes(...)` describe exactamente esta dinámica -- la usan
tanto `env.step()` (para muestrear un paso real) como `models.build_model` (para
construir la matriz `P` completa), así que ambas nunca pueden desincronizarse:

- **Movimiento** (`SOUTH/NORTH/EAST/WEST`): con la configuración por defecto
  (`slip_prob=0`), el resultado es único -- el taxi se mueve una celda, salvo que cruce
  el borde de la grilla o una pared interna (`env.internal_walls`, que por defecto solo
  bloquean `EAST`/`WEST`), en cuyo caso se queda en la misma celda. Con `slip_prob > 0`
  (ver más abajo) hay hasta 3 resultados posibles.
- **`PICKUP`/`DROPOFF` inválidos**: resultado único, ningún componente del estado
  cambia.
- **`PICKUP` válido**: resultado único, `pass_idx -> env.pass_in_taxi`.
- **`DROPOFF` válido**: resultado único -- el taxi pasa al **estado absorbente**. El
  episodio ya terminó, así que no importa en qué celda ni con qué pasajero haya quedado;
  ese estado extra (`models.mdp.StateSpace.absorbing_state`) se repite a sí mismo con
  recompensa 0 para siempre. Es la forma estándar de representar "el episodio terminó"
  dentro de las ecuaciones de Bellman, que están pensadas para horizonte infinito.

### Estocasticidad

`MilanTaxiEnv` acepta `slip_prob` (default `0.0`): con esa probabilidad, una acción de
movimiento "resbala" hacia una de las dos direcciones perpendiculares a la intencionada
(mitad de probabilidad cada una) en vez de ir hacia donde se le pidió; `PICKUP`/`DROPOFF`
nunca resbalan. `slip_prob=0.0` es el ambiente determinista de arriba. Este notebook usa
la configuración por defecto; la sección "Parámetros configurables" de abajo muestra
cómo activarla.

### Horizonte

`env.step()` trunca a `env.horizon` pasos como respaldo (default $H=100$), pero con
`DROPOFF` terminando el episodio casi nunca hace falta: un episodio normal dura muchos
menos pasos. El horizonte no participa en la construcción de $P, R$ ni en las ecuaciones
de Bellman.

### $\gamma$ (factor de descuento)

$\gamma = 0.99$ en las configs (`configs/*.yaml`), aunque la tarea ya sea episódica.
Motivo: una política todavía no óptima puede ser **impropia** (dar vueltas para siempre
sin llegar nunca a hacer `DROPOFF`). Con $\gamma=1$ el valor de esa política diverge a
$-\infty$ y la solución exacta por álgebra lineal deja de tener sentido. Con $\gamma<1$
eso no puede pasar, porque cualquier recompensa futura pesa cada vez menos.

`models.build_model(env)` arma `P`/`R` (tamaño `(n_states+1) x 6 x (n_states+1)` --
$501 \times 6 \times 501$ con la configuración por defecto) recorriendo los estados
reales de `env` y reusando `env._movement_outcomes(...)` para cada `(s, a)`, para que el
modelo nunca se desincronice de lo que hace `env.step()` (se valida esto más abajo).

### Parámetros configurables (grilla, recompensas, estocasticidad)

Todo lo anterior es la configuración por defecto. `MilanTaxiEnv` acepta estos kwargs,
reenviados tal cual desde el bloque `env:` de un YAML en `configs/` (vía `make_env`):
`num_rows`, `num_cols`, `locs`, `internal_walls`, `step_reward`, `invalid_reward`,
`dropoff_reward`, `slip_prob`, `horizon`. Por ejemplo, para una grilla más grande con
transición estocástica:

```python
env_stoch = make_env(slip_prob=0.1)
env_bigger = make_env(num_rows=7, num_cols=7, locs=[(0, 0), (0, 6), (6, 0), (6, 6)], internal_walls=[])
```

El renderizado visual (`env.render()`, pygame) solo funciona con la grilla y landmarks
originales de Milán -- cualquier otra configuración usa `env.render_text()` en su lugar
(ver docstring de `envs/milan_taxi.py`). La última sección del notebook explica cómo
correr y comparar estos experimentos en detalle.



Creamos el ambiente con la configuración por defecto, reseteamos para obtener el primer
estado, e imprimimos el estado inicial y el render en texto (para confirmar visualmente
que el estado impreso corresponde a lo que se ve en la grilla).


In [2]:
env = make_env()
state, info = env.reset()
print("estado inicial:", state)
print(env.render())

estado inicial: (4, 4, 1, 3)
[[[110 109 106]
  [110 109 106]
  [124 122 122]
  ...
  [108 111 109]
  [108 111 109]
  [118 119 119]]

 [[110 109 106]
  [110 109 106]
  [124 122 122]
  ...
  [108 111 109]
  [108 111 109]
  [118 119 119]]

 [[114 116 115]
  [114 116 115]
  [126 127 126]
  ...
  [112 113 111]
  [112 113 111]
  [118 117 115]]

 ...

 [[116 115 116]
  [116 115 116]
  [106 107 108]
  ...
  [113 115 114]
  [113 115 114]
  [117 114 117]]

 [[116 115 116]
  [116 115 116]
  [106 107 108]
  ...
  [113 115 114]
  [113 115 114]
  [117 114 117]]

 [[115 112 112]
  [115 112 112]
  [119 119 117]
  ...
  [123 119 118]
  [123 119 118]
  [114 114 117]]]


## Validación visual (pygame + animación)

Antes de tocar Dynamic Programming, conviene *ver* que el ambiente se comporta como se
espera. La celda de abajo ejecuta 30 pasos con una política aleatoria y anima los
frames: `env.render()` genera un frame RGB con `pygame`, `utils.animate_frames` los
anima en el notebook.


In [3]:
from rl_project.utils import animate_frames


class RandomPolicyDemo:
    def get_action(self, state):
        return np.random.randint(6)


np.random.seed(0)
frames = [env.render()]
state = env.state
for _ in range(30):
    action = RandomPolicyDemo().get_action(state)
    state, reward, terminated, truncated, info = env.step(action)
    frames.append(env.render())
    if terminated or truncated:
        break

animate_frames(frames)

## Política aleatoria de referencia

Antes de resolver el MDP con Policy Iteration / Value Iteration, conviene tener una
política aleatoria como punto de comparación ("baseline") contra el cual medir la
política óptima más adelante.


In [4]:
class RandomPolicy:
    def __init__(self, n_actions=6):
        self.n_actions = n_actions

    def get_action(self, state):
        return np.random.randint(self.n_actions)


np.random.seed(0)
policy = RandomPolicy()

states, actions, rewards = rollout(env, policy, render=False)
print(f"episodio de {len(actions)} pasos, retorno total = {sum(rewards)}")

episodio de 100 pasos, retorno total = -388


In [5]:
mean_return, margin = evaluate_policy(env, policy, n_runs=100, discount_factor=0.99, display_result=True)

J^pi_gamma in [-254.73, -244.45] w.p. 0.95  (n_runs=100)


## Validación del modelo (P, R)

Las celdas de abajo construyen `P, R` y corren dos chequeos antes de pasar a Policy
Iteration / Value Iteration: que `P` sea una distribución de probabilidad válida (cada
fila suma 1), y que el modelo se pueda resolver por álgebra lineal exacta para una
política fija (chequeo cruzado independiente de PI/VI). `models.build_model` recorre
los estados reales de `env` (500 con la configuración por defecto), agrega el estado
absorbente, y arma `P, R` reusando `env._movement_outcomes(...)` -- la misma
distribución que usa `env.step()` para muestrear (ver la sección **Ambiente**, arriba).


In [6]:
from rl_project.models import build_model

P, R, state_space = build_model(env)
print(f"P: {P.shape}, R: {R.shape}")  # (n_states+1, 6, n_states+1) -- incluye el estado absorbente

row_sums = P.sum(axis=2)
print(f"max |sum(P[s,a,:]) - 1| = {np.max(np.abs(row_sums - 1.0)):.2e}")

# El estado absorbente se repite a sí mismo con recompensa 0 para siempre.
assert np.all(P[state_space.absorbing_state, :, state_space.absorbing_state] == 1.0)
assert np.all(R[state_space.absorbing_state, :] == 0.0)


P: (501, 6, 501), R: (501, 6)
max |sum(P[s,a,:]) - 1| = 0.00e+00


In [7]:
# Solución exacta (I - gamma P^pi) V = R^pi para la política aleatoria, resolviendo el
# sistema lineal directamente -- sirve como chequeo cruzado independiente de PI/VI.
GAMMA = 0.99
n_states_model, n_actions = R.shape
pi_random = np.ones((n_states_model, n_actions)) / n_actions

P_pi = np.einsum('sa,sat->st', pi_random, P)
R_pi = np.einsum('sa,sa->s', pi_random, R)
V_random = np.linalg.solve(np.eye(n_states_model) - GAMMA * P_pi, R_pi)

print(f"V^random: min={V_random.min():.1f}  max={V_random.max():.1f}  mean={V_random.mean():.1f}")
print(f"V^random(estado absorbente) = {V_random[state_space.absorbing_state]:.4f}  (debe ser ~0)")


V^random: min=-395.7  max=0.0  mean=-356.1
V^random(estado absorbente) = 0.0000  (debe ser ~0)


## Policy Iteration y Value Iteration

Con `P, R` ya construidos y validados, resolvemos el MDP con los dos algoritmos de
Programación Dinámica del taller (`src/rl_project/agents/dynamic_programming.py`).
Ambos deberían converger al mismo $V^*$ y a la misma política óptima -- si no
coinciden, hay un error en la implementación de alguno de los dos.


In [8]:
from rl_project.agents import PolicyIterationAgent, ValueIterationAgent

pi_agent = PolicyIterationAgent(gamma=GAMMA, theta=1e-8)
V_star, pi_star = pi_agent.solve(P, R)
print(f"Policy Iteration: convergio en {pi_agent.n_iterations} iteraciones")

vi_agent = ValueIterationAgent(gamma=GAMMA, theta=1e-8)
V_star_vi, pi_star_vi = vi_agent.solve(P, R)
print(f"Value Iteration:  convergio en {vi_agent.n_iterations} sweeps")

print(f"max |V_PI - V_VI| = {np.max(np.abs(V_star - V_star_vi)):.2e}")
print(f"misma politica greedy: {np.array_equal(pi_star.argmax(1), pi_star_vi.argmax(1))}")

Policy Iteration: convergio en 3 iteraciones
Value Iteration:  convergio en 19 sweeps
max |V_PI - V_VI| = 0.00e+00
misma politica greedy: True


### Validación visual: el taxi resolviendo el problema

`TabularPolicy` envuelve la política óptima (`pi_star`, one-hot) para poder usarla con
`evaluation.rollout` exactamente igual que la política aleatoria -- así se puede *ver*
que la política encontrada realmente recoge al pasajero y lo deja en su destino, y no
solo confiar en que $V^*$ dio un número razonable.


In [9]:
from rl_project.policies import TabularPolicy

optimal_policy = TabularPolicy(pi_star, state_space.state_to_idx)

frames, states, actions, rewards = rollout(env, optimal_policy, render=True, seed=0)
print(f"episodio de {len(actions)} pasos, retorno total = {sum(rewards)}")

animate_frames(frames)


episodio de 7 pasos, retorno total = 14


In [10]:
# Comparación cuantitativa (no solo V* teórico, sino simulando episodios reales):
# política óptima vs. política aleatoria, con el mismo número de corridas.
mean_opt, margin_opt = evaluate_policy(env, optimal_policy, n_runs=100, discount_factor=GAMMA, display_result=True)
mean_rand, margin_rand = evaluate_policy(env, RandomPolicy(), n_runs=100, discount_factor=GAMMA, display_result=True)


J^pi_gamma in [5.96, 6.99] w.p. 0.95  (n_runs=100)
J^pi_gamma in [-256.64, -245.79] w.p. 0.95  (n_runs=100)


## Próximos experimentos

Todo lo necesario para experimentar ya está implementado -- no hace falta tocar código,
solo cambiar los kwargs de `env:` (grilla, `slip_prob`, recompensas) en un YAML nuevo
dentro de `configs/`, o pasarlos directamente a `make_env(...)` acá mismo en el notebook.
Tres experimentos concretos:

1. **Grilla más grande**: `make_env(num_rows=7, num_cols=7, locs=[...], internal_walls=[])`.
2. **Transición estocástica**: `make_env(slip_prob=0.1)` (o el valor que se quiera).
3. **Recompensas distintas**: `make_env(step_reward=-2)`, `make_env(dropoff_reward=5)`, etc.

Para cada uno, `build_model` + `PolicyIterationAgent`/`ValueIterationAgent` sobre el
`env` modificado da directamente `n_iterations`, `V*` y la política óptima para comparar
contra el caso por defecto (celdas 10-13 de este notebook). `cli.py train/experiment/
visualize` hacen lo mismo desde una config guardada, para dejar registro en `runs/`.


In [15]:
env_custom = make_env(
    step_reward=1,      # movimiento
    pickup_reward=1,    # PICKUP válido
    invalid_reward=-50,  # PICKUP o DROPOFF inválido
    dropoff_reward=50,   # DROPOFF exitoso
)

P_c, R_c, ss_c = build_model(env_custom)

pi_agent_c = PolicyIterationAgent(gamma=0.99, theta=1e-8)
V_c, policy_c = pi_agent_c.solve(P_c, R_c)
vi_agent_c = ValueIterationAgent(gamma=0.99, theta=1e-8)
V_c_vi, policy_c_vi = vi_agent_c.solve(P_c, R_c)

print(f"PI: {pi_agent_c.n_iterations} iteraciones, VI: {vi_agent_c.n_iterations} sweeps")
print(f"V* min/max/mean: {V_c.min():.1f} / {V_c.max():.1f} / {V_c.mean():.1f}")
print(f"% estados con acción distinta a la baseline: {(policy_c.argmax(1) != pi_star.argmax(1)).mean()*100:.2f}%")

PI: 4 iteraciones, VI: 1766 sweeps
V* min/max/mean: 0.0 / 100.0 / 99.8
% estados con acción distinta a la baseline: 63.87%
